# HEPMC Fireworks: Sonifying and Visualizing LHC Collision Events



<CENTER><img src="https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/images/ATLASOD.gif?raw=1" style="width:50%"></CENTER>

This notebook turns individual proton&ndash;proton collision events, stored in
[HEPMC](https://cds.cern.ch/record/684090/) format, into short audio-visual
clips: the initial hard collision plays like a firework's **pop**, and the
parton shower and hadronization that follows plays out as a **crackle** of
notes, colored and voiced by particle type, while the event visually grows
outward from the collision point.

We fetch example event files directly from the ATLAS Open Data event-generation
release using the [`atlasopenmagic`](https://opendata.atlas.cern/docs/data/atlasopenmagic)
package, following the same pattern as the official
[`OpenEvgenTutorial.ipynb`](https://github.com/atlas-outreach-data-tools/notebooks-collection-opendata/blob/master/for-research/OpenEvgenTutorial.ipynb)
notebook. Two example datasets (DSIDs `410470` and `700322`) are used below,
but any HEPMC file from ATLAS Open Data or elsewhere will work.

**What's inside:**
1. Fetching HEPMC files for a given ATLAS dataset ID (DSID)
2. A short, roughly-explained sonification/visualization engine
3. Running it on example events and playing the results inline

If you use these samples in other work, please remember to cite them &mdash;
see the [Event Generation Data](https://opendata.atlas.cern/docs/data/for_research/evgen_data)
page for details.

## Setup

`atlasopenmagic` gives us metadata and streaming URLs for ATLAS Open Data
samples; `pyhepmc` reads the HEPMC files; the rest is standard scientific
Python plus `ffmpeg` for muxing the final mp4.

In [ ]:
%pip install -q atlasopenmagic pyhepmc numpy matplotlib requests
# ffmpeg is required to mux audio and video; most notebook platforms
# (Binder, Colab, SWAN) already have it, but this makes sure it's there.
import shutil
if shutil.which("ffmpeg") is None:
    import sys
    if sys.platform.startswith("linux"):
        !apt-get -qq update && apt-get -qq install -y ffmpeg
    else:
        print("Please install ffmpeg manually and make sure it's on your PATH.")

## 1. Getting HEPMC files from ATLAS Open Data

The event-generation samples live in a couple of `atlasopenmagic` "releases"
(one per collision energy). We look the right one up by name rather than
hard-coding it, since release tags can change over time. Some datasets are available in both releases; this will favor the 13.6 TeV release when it's available.

In [ ]:
import atlasopenmagic as atom

print(atom.available_releases())

In [ ]:
# Run-3-like (13.6 TeV) samples are used here
atom.set_release("2025r-evgen-13p6tev")
DSIDS = ["601229", "700322"]
for dsid in DSIDS:
    meta = atom.get_metadata(dsid)
    print(f"DSID {dsid}")
    print("  physics_short:", meta.get("physics_short"))
    print("  CoM energy:   ", meta.get("CoMEnergy"), "GeV")
    print("  cross section:", meta.get("cross_section_pb"), "pb")
    print()

`410470` is the standard ttbar (top quark pair) sample used across many ATLAS
Open Data tutorials; `700322` is a standard Z-boson sample with Z-boson decays to an electron&ndash;positron pair. Feel free to try things out with your favorite samples. The [metadata tutorial](https://opendata.atlas.cern/docs/13TeV25Doc/Concepts#accessing-metadata$0) has many examples of how to find samples. You might find that differences between event generators (e.g. between Pythia and Sherpa) are far greater than the differences between events, or even between processes.

Next we'll download a couple of files from each dataset. You can also upload a file of your choosing to where ever you're processing the data. Depending on your connection and the sample you've chosen, this could take a moment, because some of the files are rather large. We will also do a little adjusting: the open data files have slightly different names or types compared to what `pyhepmc` expects.

In [ ]:
import os
import tarfile
import requests

DATA_DIR = "hepmc_data"
os.makedirs(DATA_DIR, exist_ok=True)


def _download(url, dest_dir=DATA_DIR):
    local_path = os.path.join(dest_dir, os.path.basename(url.split("?")[0]))
    if os.path.exists(local_path):
        return local_path
    print(f"Downloading {url} ...")
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    if local_path.endswith(".tar.gz.1"):
        import tarfile
        with tarfile.open(local_path, mode="r:gz") as tar_file:
            hepmc_file = tar_file.getmembers()[0]
            tar_file.extract(hepmc_file, filter="data")
        local_path = hepmc_file.name
    elif local_path.endswith(".gz.1"):
        adjusted_path = local_path.replace(".gz.1",".gz")
        os.rename(local_path,adjusted_path)
        local_path = adjusted_path
    return local_path


def fetch_hepmc_file(dsid, dest_dir=DATA_DIR, file_index=0):
    # Download one HEPMC file for a DSID and return a path pyhepmc can open.
    # pyhepmc.open() reads gzip-compressed files natively, so a plain
    # `.hepmc.gz` doesn't need to be unpacked here -- only tar archives do.
    rel, _ = atom.get_metadata(dsid)
    urls = atom.get_urls(dsid, protocol="https", cache=False)
    if not urls:
        raise RuntimeError(f"No files found for DSID {dsid} in release {rel}")

    local_path = _download(urls[file_index], dest_dir)

    if local_path.endswith(".tar.gz") or local_path.endswith(".tgz"):
        with tarfile.open(local_path, "r:gz") as tf:
            members = [m for m in tf.getmembers() if m.isfile()]
            tf.extractall(dest_dir)
        return os.path.join(dest_dir, members[0].name)
    return local_path


hepmc_path_ttbar = fetch_hepmc_file("601229")
hepmc_path_zee = fetch_hepmc_file("700322")
print(hepmc_path_ttbar)
print(hepmc_path_zee)

> **Using your own file instead:** if you already have a HEPMC file (for
> example a small sample you generated yourself, or one downloaded by hand),
> just skip the cells above and set `hepmc_path_ttbar = "/path/to/your.hepmc"`.
> Everything below only needs a path that `pyhepmc.open()` can read.

## 2. The sonification / visualization engine

Very roughly, here's what the code in this section does:

- **Build a tree.** Each particle's parents/children (straight from the
  file's vertex structure) give us a graph; a breadth-first search from the
  incoming beam particles gives every particle a shower "generation" (depth).
- **Map depth &rarr; time.** An S-curve turns generation depth into a birth
  time: the first, hardest splittings ease in shortly after an initial
  "pop", the bulk of the shower fills the middle of the clip, and the many
  late, soft particles from hadronization bunch up into a dense "crackle"
  near the end.
- **Map depth &rarr; position.** Particles are laid out as a branching tree in
  2D: each one sits one step beyond its parent, along its own momentum
  direction, so the whole event grows outward from the middle like a
  firework burst.
- **Voice each particle.** Quarks, gluons, photons, leptons, and hadrons each
  get a distinct synthesized timbre; pitch is set by (log) energy, quantized
  to a pentatonic scale so overlapping notes stay consonant. Neutrinos are
  silent, since they escape undetected.
- **Render.** Audio is mixed to a WAV; a silent MP4 is drawn frame-by-frame
  with matplotlib (bright flashes on birth, fading trails behind them); the
  two are muxed together with `ffmpeg`.

The code below is essentially a plain Python module split across a few
cells &mdash; run them in order once, then jump to Section 3 to use it.

In [ ]:
import math
import os
import subprocess
import tempfile
from collections import deque
from dataclasses import dataclass, field

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.animation import FFMpegWriter

import pyhepmc

In [ ]:
# --- Particle family classification & voicing --------------------------

FAMILY_QUARK = "quark"
FAMILY_GLUON = "gluon"
FAMILY_PHOTON = "photon"
FAMILY_LEPTON = "lepton"
FAMILY_HADRON = "hadron"

QUARK_PIDS = {1, 2, 3, 4, 5, 6}
LEPTON_PIDS = {11, 12, 13, 14, 15, 16}
NEUTRINO_PIDS = {12, 14, 16}

FAMILY_COLOR = {
    FAMILY_QUARK: (1.00, 0.35, 0.35),   # red
    FAMILY_GLUON: (0.75, 0.75, 0.80),   # smoky grey
    FAMILY_PHOTON: (1.00, 0.92, 0.45),  # bright gold
    FAMILY_LEPTON: (0.40, 0.80, 1.00),  # cyan
    FAMILY_HADRON: (0.55, 1.00, 0.60),  # green
}

# Base register (Hz) and "hang time" (seconds a spark stays visually/aurally
# alive after birth) per family.
FAMILY_REGISTER = {
    FAMILY_QUARK: 110.0,
    FAMILY_GLUON: 90.0,
    FAMILY_PHOTON: 660.0,
    FAMILY_LEPTON: 330.0,
    FAMILY_HADRON: 220.0,
}
FAMILY_HANG = {
    FAMILY_QUARK: 0.45,
    FAMILY_GLUON: 0.30,
    FAMILY_PHOTON: 0.85,
    FAMILY_LEPTON: 0.70,
    FAMILY_HADRON: 0.35,
}

# Major-pentatonic semitone offsets keep simultaneous notes consonant.
PENTATONIC = [0, 2, 4, 7, 9]


def classify(pid):
    apid = abs(pid)
    if apid == 21:
        return FAMILY_GLUON
    if apid == 22:
        return FAMILY_PHOTON
    if apid in QUARK_PIDS:
        return FAMILY_QUARK
    if apid in LEPTON_PIDS:
        return FAMILY_LEPTON
    return FAMILY_HADRON


# Map an energy (GeV) onto a quantized pentatonic pitch around a base
# register, using a log scale so both very soft and very hard particles
# land in a sane audible range.
def energy_to_freq(e_gev, register, octave_span=2.2):
    e = max(e_gev, 1e-3)
    logscale = math.log2(e)
    norm = (logscale + 6.0) / 14.0
    norm = min(max(norm, 0.0), 1.0)
    n_steps = len(PENTATONIC) * octave_span
    idx = norm * (n_steps - 1)
    i = int(round(idx))
    octave, step = divmod(i, len(PENTATONIC))
    semitone = PENTATONIC[step] + 12 * octave
    return register * (2.0 ** (semitone / 12.0))

In [ ]:
# --- Graph construction ---------------------------------------------------

@dataclass
class Node:
    id: int
    pid: int
    status: int
    E: float
    px: float
    py: float
    pz: float
    parent_ids: list = field(default_factory=list)
    children_ids: list = field(default_factory=list)
    parent_id: int = None
    depth: int = 0
    family: str = ""
    phi: float = 0.0
    pt: float = 0.0
    birth_time: float = 0.0
    x: float = 0.0
    y: float = 0.0
    px_pos: float = 0.0  # parent x (for drawing branch line)
    py_pos: float = 0.0


def build_graph(event):
    nodes = {}
    for p in event.particles:
        mom = p.momentum
        nodes[p.id] = Node(
            id=p.id, pid=p.pid, status=p.status,
            E=mom.e, px=mom.px, py=mom.py, pz=mom.pz,
            parent_ids=[x.id for x in p.parents],
            children_ids=[x.id for x in p.children],
        )

    roots = [nid for nid, n in nodes.items() if not n.parent_ids]

    # BFS shortest-path depth from the incoming beam particles.
    depth = {r: 0 for r in roots}
    q = deque(roots)
    while q:
        cur = q.popleft()
        d = depth[cur]
        for ch in nodes[cur].children_ids:
            nd = d + 1
            if ch not in depth or nd < depth[ch]:
                depth[ch] = nd
                q.append(ch)

    for nid, n in nodes.items():
        n.depth = depth.get(nid, 0)
        n.family = classify(n.pid)
        n.pt = math.hypot(n.px, n.py)
        n.phi = math.atan2(n.py, n.px)

    # Primary parent = the immediate-generation (max-depth) parent, so depth
    # strictly increases along the chosen tree edges; ties broken by energy.
    for nid, n in nodes.items():
        if n.parent_ids:
            n.parent_id = max(
                n.parent_ids, key=lambda pid: (nodes[pid].depth, nodes[pid].E)
            )

    return nodes, roots

In [ ]:
# --- Time and position mapping -------------------------------------------

# S-curve time map: depth=0 is the pop at t=0; early generations ease
# in shortly after; late, soft generations compress into a crackle near
# the tail. Small jitter smooths each depth's simultaneous chord into a
# spray of nearly-but-not-quite-simultaneous hits.
def compute_times(nodes, duration, pop_time, rng, jitter_frac=0.55):
    maxdepth = max((n.depth for n in nodes.values()), default=1)
    maxdepth = max(maxdepth, 1)

    d0 = 0.32 * maxdepth
    w = max(0.14 * maxdepth, 1.0)

    def sigmoid(d):
        return 1.0 / (1.0 + math.exp(-(d - d0) / w))

    lo, hi = sigmoid(0), sigmoid(maxdepth)
    span = (hi - lo) or 1.0

    depth_time = {}
    for d in range(maxdepth + 1):
        frac = (sigmoid(d) - lo) / span
        depth_time[d] = pop_time + (duration - pop_time) * frac

    for n in nodes.values():
        if n.depth == 0:
            n.birth_time = 0.0
            continue
        t = depth_time[n.depth]
        prev_t = depth_time.get(n.depth - 1, 0.0)
        next_t = depth_time.get(n.depth + 1, duration)
        spread = jitter_frac * min(t - prev_t, next_t - t)
        t += rng.uniform(-spread, spread)
        n.birth_time = min(max(t, pop_time * 0.5), duration - 0.01)


# Lay particles out as a branching tree: each sits one step beyond its
# parent, in the direction of its own momentum, so the event grows
# outward from the collision point like a firework burst.
def compute_positions(nodes, roots, step_base=0.975):
    for r in roots:
        nodes[r].x = 0.0
        nodes[r].y = 0.0

    order = sorted(nodes.values(), key=lambda n: n.depth)
    max_pt = max((n.pt for n in nodes.values()), default=1.0) or 1.0
    for n in order:
        if n.depth == 0:
            continue
        parent = nodes.get(n.parent_id)
        if parent is None:
            n.x, n.y = 0.0, 0.0
            continue
        step_scale = 0.35 + 0.65 * (math.log1p(n.pt) / math.log1p(max_pt))
        step = step_base * step_scale
        n.x = parent.x + step * math.cos(n.phi)
        n.y = parent.y + step * math.sin(n.phi)
        n.px_pos, n.py_pos = parent.x, parent.y

In [ ]:
# --- Audio synthesis -------------------------------------------------------

def _env(n, sr, attack, decay):
    t = np.arange(n) / sr
    att_n = max(1, int(attack * sr))
    env = np.ones(n)
    ramp = np.linspace(0.0, 1.0, min(att_n, n))
    env[: len(ramp)] = ramp
    env *= np.exp(-t / max(decay, 1e-4))
    return env


def synth_quark(freq, amp, dur, sr, rng):
    n = int(dur * sr)
    t = np.arange(n) / sr
    wave = 0.7 * np.sin(2 * np.pi * freq * t) + 0.3 * np.sin(2 * np.pi * 2 * freq * t)
    env = _env(n, sr, attack=0.005, decay=dur * 0.35)
    return amp * wave * env


def synth_gluon(freq, amp, dur, sr, rng):
    n = max(int(dur * sr), 8)
    noise = rng.uniform(-1, 1, n)
    spec = np.fft.rfft(noise)
    freqs = np.fft.rfftfreq(n, d=1 / sr)
    bw = freq * 0.6 + 40
    mask = np.exp(-0.5 * ((freqs - freq) / bw) ** 2)
    filtered = np.fft.irfft(spec * mask, n)
    peak = np.max(np.abs(filtered)) + 1e-9
    filtered /= peak
    env = _env(n, sr, attack=0.001, decay=dur * 0.22)
    return amp * filtered * env


def synth_photon(freq, amp, dur, sr, rng):
    n = int(dur * sr)
    t = np.arange(n) / sr
    vibrato = 1.0 + 0.004 * np.sin(2 * np.pi * 5.0 * t)
    wave = np.sin(2 * np.pi * freq * vibrato * t) + 0.15 * np.sin(2 * np.pi * 3 * freq * t)
    env = _env(n, sr, attack=0.012, decay=dur * 0.55)
    return amp * wave * env


def synth_lepton(freq, amp, dur, sr, rng):
    n = int(dur * sr)
    t = np.arange(n) / sr
    wave = np.sin(2 * np.pi * freq * t) + 0.2 * np.sin(2 * np.pi * 0.5 * freq * t)
    env = _env(n, sr, attack=0.008, decay=dur * 0.45)
    return amp * wave * env


def synth_hadron(freq, amp, dur, sr, rng):
    n = int(dur * sr)
    t = np.arange(n) / sr
    tone = np.sin(2 * np.pi * freq * t)
    noise = rng.uniform(-1, 1, n)
    click_env = np.exp(-t / 0.008)
    env = _env(n, sr, attack=0.001, decay=dur * 0.2)
    return amp * (0.7 * tone * env + 0.3 * noise * click_env)


# The initial hard-collision 'pop': a low thump with a fast pitch
# drop, layered with a short broadband burst.
def synth_pop(amp, dur, sr, rng):
    n = int(dur * sr)
    t = np.arange(n) / sr
    f0, f1 = 140.0, 45.0
    freq_t = f1 + (f0 - f1) * np.exp(-t / 0.05)
    phase = 2 * np.pi * np.cumsum(freq_t) / sr
    thump = np.sin(phase) * np.exp(-t / 0.25)
    noise = rng.uniform(-1, 1, n) * np.exp(-t / 0.04)
    return amp * (0.85 * thump + 0.5 * noise)


SYNTH_FOR_FAMILY = {
    FAMILY_QUARK: synth_quark,
    FAMILY_GLUON: synth_gluon,
    FAMILY_PHOTON: synth_photon,
    FAMILY_LEPTON: synth_lepton,
    FAMILY_HADRON: synth_hadron,
}


def render_audio(nodes, duration, sr, min_energy_frac, rng, end_silence=0.5):
    n_samples = int(duration * sr) + sr
    buf = np.zeros(n_samples)

    audible = [n for n in nodes.values() if n.depth > 0]
    if not audible:
        return buf[: int(duration * sr)]
    max_e = max(n.E for n in audible)
    threshold = max_e * min_energy_frac

    pop_amp = 0.9
    pop = synth_pop(pop_amp, 0.5, sr, rng)
    _mix(buf, pop, 0, sr)

    for n in audible:
        if n.E < threshold:
            continue
        if n.pid in NEUTRINO_PIDS:
            continue  # neutrinos escape undetected -- silent, like real life
        e_gev = n.E / 1000.0
        freq = energy_to_freq(e_gev, FAMILY_REGISTER[n.family])
        amp = 0.5 * math.sqrt(min(n.E / max_e, 1.0)) + 0.03
        note_dur = max(0.06, FAMILY_HANG[n.family] * (0.4 + 0.6 * (n.E / max_e)))
        synth = SYNTH_FOR_FAMILY[n.family]
        note = synth(freq, amp, note_dur, sr, rng)
        _mix(buf, note, n.birth_time, sr)

    buf = buf[: int(duration * sr)]
    peak = np.max(np.abs(buf)) + 1e-9
    if peak > 1.0:
        buf = np.tanh(buf / peak * 1.4) * 0.95
    else:
        buf = buf * min(1.0, 0.9 / peak)

    # Guarantee at least `end_silence` seconds of true silence at the tail,
    # with a brief smooth ramp into it rather than an abrupt cut.
    if end_silence > 0:
        ramp_len = min(int(0.03 * sr), len(buf))
        fade_start = max(0, len(buf) - int(end_silence * sr) - ramp_len)
        ramp_end = min(fade_start + ramp_len, len(buf))
        if ramp_end > fade_start:
            buf[fade_start:ramp_end] *= np.linspace(1.0, 0.0, ramp_end - fade_start)
        buf[ramp_end:] = 0.0

    return buf


def _mix(buf, note, t0, sr):
    start = int(t0 * sr)
    end = start + len(note)
    if start >= len(buf):
        return
    end = min(end, len(buf))
    buf[start:end] += note[: end - start]


def write_wav(path, samples, sr):
    import wave
    samples = np.clip(samples, -1.0, 1.0)
    pcm = (samples * 32767.0).astype(np.int16)
    with wave.open(path, "w") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sr)
        w.writeframes(pcm.tobytes())

In [ ]:
# --- Video rendering ---------------------------------------------------

def render_video(nodes, duration, fps, out_path, min_energy_frac, dpi=110,
                  figsize=(6, 6), end_fade=1.5):
    audible = [n for n in nodes.values() if n.depth > 0]
    max_e = max((n.E for n in audible), default=1.0)
    threshold = max_e * min_energy_frac
    shown = [n for n in audible if n.E >= threshold]

    xs = np.array([n.x for n in shown])
    ys = np.array([n.y for n in shown])
    lim = max(1.0, np.max(np.abs(np.concatenate([xs, ys]))) * 0.7) if len(shown) else 1.0

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi, facecolor="black")
    ax.set_facecolor("black")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_aspect("equal")

    segments, seg_colors, seg_birth = [], [], []
    for n in shown:
        segments.append([(n.px_pos, n.py_pos), (n.x, n.y)])
        seg_colors.append(FAMILY_COLOR[n.family])
        seg_birth.append(n.birth_time)
    seg_birth = np.array(seg_birth)

    lc = LineCollection(segments, colors=seg_colors, linewidths=1.1)
    lc.set_alpha(0.0)
    ax.add_collection(lc)

    scat = ax.scatter([], [], s=[], c=[], edgecolors="none")

    n_frames = int(duration * fps) + 1
    hang = np.array([FAMILY_HANG[n.family] for n in shown])
    birth = np.array([n.birth_time for n in shown])
    ex = np.array([n.E / max_e for n in shown])
    colors = np.array([FAMILY_COLOR[n.family] for n in shown])
    xs = np.array([n.x for n in shown])
    ys = np.array([n.y for n in shown])

    fade_start = max(0.0, duration - end_fade)

    def frame(i):
        t = i / fps

        line_alpha = np.where(
            t < seg_birth, 0.0,
            np.where(t < seg_birth + 0.15, 0.75 * (t - seg_birth) / 0.15, 0.2),
        )
        lc.set_alpha(np.clip(line_alpha, 0.0, 0.75).tolist())

        age = t - birth
        alive = (age >= 0) & (age <= hang)
        if not np.any(alive):
            scat.set_offsets(np.empty((0, 2)))
            return (lc, scat)

        if end_fade > 0 and t > fade_start:
            circle_fade = max(0.0, 1.0 - (t - fade_start) / end_fade)
        else:
            circle_fade = 1.0

        frac = age[alive] / hang[alive]
        flash = np.exp(-3.0 * frac)
        sizes = (10 + 260 * ex[alive]) * (0.3 + 0.7 * flash)
        alphas = np.clip((0.25 + 0.75 * flash) * circle_fade, 0.0, 1.0)
        rgba = np.concatenate([colors[alive], alphas.reshape(-1, 1)], axis=1)
        scat.set_offsets(np.column_stack([xs[alive], ys[alive]]))
        scat.set_sizes(sizes)
        scat.set_color(rgba)
        return (lc, scat)

    writer = FFMpegWriter(fps=fps, codec="libx264", bitrate=4000,
                           extra_args=["-pix_fmt", "yuv420p"])
    with writer.saving(fig, out_path, dpi=dpi):
        for i in range(n_frames):
            frame(i)
            writer.grab_frame(facecolor="black")
    plt.close(fig)

In [ ]:
# --- Orchestration ---------------------------------------------------------

# Render a single pyhepmc GenEvent to an mp4 with sound at `out_path`.
def render_hepmc_event(event, out_path, duration=7.0, fps=30, sample_rate=44100,
                        min_energy_frac=0.0015, spread=0.975, end_fade=1.5,
                        end_silence=0.5, seed=0):
    rng = np.random.default_rng(seed)
    nodes, roots = build_graph(event)
    active_duration = max(1.0, duration - end_silence)
    compute_times(nodes, active_duration, pop_time=0.35, rng=rng)
    compute_positions(nodes, roots, step_base=spread)

    with tempfile.TemporaryDirectory() as tmp:
        wav_path = os.path.join(tmp, "audio.wav")
        video_only_path = os.path.join(tmp, "video.mp4")

        print(f"synthesizing audio ({len(nodes)} particles)...")
        audio = render_audio(nodes, duration, sample_rate, min_energy_frac, rng,
                              end_silence=end_silence)
        write_wav(wav_path, audio, sample_rate)

        print("rendering video...")
        render_video(nodes, duration, fps, video_only_path, min_energy_frac,
                     end_fade=end_fade)

        print(f"muxing audio + video -> {out_path}")
        cmd = ["ffmpeg", "-y", "-i", video_only_path, "-i", wav_path,
               "-c:v", "copy", "-c:a", "aac", "-shortest", out_path]
        subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
    return out_path

## 3. Run it

We take the first event from each downloaded file and render it. Feel free
to change `skip_events` to look at a different event in the file, or to
loop over several.

In [ ]:
from IPython.display import Video

def render_first_event(hepmc_path, out_name, skip_events=0, **kwargs):
    with pyhepmc.open(hepmc_path) as f:
        for i, event in enumerate(f):
            if i < skip_events:
                continue
            return render_hepmc_event(event, out_name, **kwargs)
    raise RuntimeError(f"File has fewer than {skip_events + 1} events.")

In [ ]:
out_ttbar = render_first_event(hepmc_path_ttbar, "ttbar_410470_event0.mp4")
Video(out_ttbar, embed=True)

In [ ]:
out_zee = render_first_event(hepmc_path_zee, "dsid700322_event0.mp4")
Video(out_zee, embed=True)

### Tuning it

A few parameters worth playing with, all keyword arguments to
`render_hepmc_event`:

| Parameter | What it does |
|---|---|
| `duration` | Total clip length in seconds (5&ndash;10 is a good range) |
| `min_energy_frac` | Drops particles below this fraction of the event's hardest particle's energy, to keep very soft radiation from turning into an undifferentiated hiss |
| `spread` | Radial step size between generations in the visualization &mdash; bigger spreads the event out more |
| `end_fade` | Seconds over which the bright particle bursts fade out at the end, leaving only the branch trails |
| `end_silence` | Seconds of guaranteed silence at the very end of the clip |
| `seed` | RNG seed for the noise-based instruments, for reproducibility |

A command-line version of this same engine (for batch-processing many
events to files instead of playing them inline) is available on request.

<div class="alert alert-block alert-info">
We welcome your feedback on this notebook or any of our other materials! Please <a href="https://forms.gle/zKBqS1opAHHemv9U7">fill out this survey</a> to let us know how we're doing, and you can enter a raffle to win some <a href="https://atlas-secretariat.web.cern.ch/merchandise">ATLAS merchandise<a>!
</div>